# 02. Score Quality

`papers_raw.csv`를 받아 정량 퀄리티 점수를 매깁니다.

**점수 산식 (단순 가중합)**:
- 인용수 백분위 (해당 검색 결과 안에서) × 0.30
- **연식 보정 인용수**(citations_per_year) 백분위 × 0.30  ← 2014년 100인용 vs 2024년 50인용 같은 불공정 보정
- 최신성 (FROM_YEAR → 0.0, TO_YEAR → 1.0 선형) × 0.20
- 저널/소스 보유 여부 × 0.10
- abstract 충분도 (200자 이상이면 1) × 0.10

이 점수는 **후보 좁히기를 위한 정량 신호**일 뿐입니다. 정성 평가는 `reference_quality_check(rr)` 프롬프트로 Claude에 맡기세요.

> ⚠️ `citations_per_year` 컬럼은 **노트북 01의 개선판이 만들어준 컬럼**입니다. 옛 papers_raw.csv를 쓰면 자동 계산으로 폴백합니다.


In [ ]:
!pip install -r ../../../requirements.txt

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path('../data')
df = pd.read_csv(DATA_DIR / 'papers_raw.csv')
print(f'{len(df)} papers loaded')
df.head()

50 papers loaded


,id,title,authors,year,venue,cited_by_count,citations_per_year,language,doi,oa_url,abstract
0,https://openalex.org/W4387835442,Generative Agents: Interactive Simulacra of Hu...,"Joon Sung Park, Joseph O’Brien, Carrie J. Cai ...",2023,NaN,1308,436.0,en,https://doi.org/10.1145/3586183.3606763,https://dl.acm.org/doi/pdf/10.1145/3586183.360...,Believable proxies of human behavior can empow...
1,https://openalex.org/W4390721568,From LLM to Conversational Agent: A Memory Enh...,"Na Liu, Liangyu Chen, Xiaoyu Tian 외 3명",2024,arXiv (Cornell University),6,3.0,en,https://doi.org/10.48550/arxiv.2401.02777,https://arxiv.org/pdf/2401.02777,This paper introduces RAISE (Reasoning and Act...
2,https://openalex.org/W4387323291,Improving Planning with Large Language Models:...,"Taylor W. Webb, Shanka Subhra Mondal, Ida Mome...",2023,arXiv (Cornell University),3,1.0,en,https://doi.org/10.48550/arxiv.2310.00194,https://arxiv.org/pdf/2310.00194,Large language models (LLMs) demonstrate impre...
3,https://openalex.org/W4414432626,A Large Language Model-Enabled Control Archite...,"Jonghan Lim, Ilya Kovalenko",2025,NaN,4,4.0,en,https://doi.org/10.1109/case58245.2025.11163802,NaN,Manufacturing environments are becoming more c...
4,https://openalex.org/W4304195432,Persona-Driven Benchmarking for Generalizable ...,Ishan Katoch,2022,arXiv (Cornell University),528,132.0,en,https://doi.org/10.48550/arxiv.2210.03629,https://arxiv.org/pdf/2210.03629,"This research paper, ""Persona-Driven Benchmark..."


In [3]:
FROM_YEAR = 2020   # 01에서 사용한 값
TO_YEAR = 2026

def score(df, from_year=FROM_YEAR, to_year=TO_YEAR):
    """5개 신호의 가중합. citations_per_year가 없으면 자동 계산."""
    cit = df['cited_by_count'].fillna(0).astype(float)
    cit_pct = cit.rank(pct=True) if cit.max() != cit.min() else pd.Series([0.5] * len(df), index=df.index)

    # 연식 보정 인용수 — 옛 csv에 컬럼이 없으면 자동 계산
    if 'citations_per_year' in df.columns:
        cpy = df['citations_per_year'].fillna(0).astype(float)
    else:
        year_for_age = df['year'].fillna(to_year).astype(float)
        age = (to_year - year_for_age).clip(lower=1)
        cpy = (cit / age).round(2)
    cpy_pct = cpy.rank(pct=True) if cpy.max() != cpy.min() else pd.Series([0.5] * len(df), index=df.index)

    year = df['year'].fillna(from_year).astype(float)
    recency = ((year - from_year) / max(to_year - from_year, 1)).clip(0, 1)

    has_venue = df['venue'].notna().astype(float)
    abs_ok = df['abstract'].fillna('').str.len().ge(200).astype(float)

    return (
        cit_pct  * 0.30
        + cpy_pct  * 0.30
        + recency  * 0.20
        + has_venue * 0.10
        + abs_ok    * 0.10
    ).round(3)

df['quality_score'] = score(df)
df_sorted = df.sort_values('quality_score', ascending=False).reset_index(drop=True)

# 표시 컬럼 — citations_per_year 추가 (있으면)
display_cols = ['title', 'year', 'cited_by_count']
if 'citations_per_year' in df_sorted.columns:
    display_cols.append('citations_per_year')
display_cols += ['venue', 'quality_score']
df_sorted[display_cols].head(15)


,title,year,cited_by_count,citations_per_year,venue,quality_score
0,Opinion Paper: “So what if ChatGPT wrote it?” ...,2023,3446,1148.67,International Journal of Information Management,0.840
1,Performance of ChatGPT on USMLE: Potential for...,2023,3435,1145.00,PLOS Digital Health,0.828
2,"Review of deep learning: concepts, CNN archite...",2021,7319,1463.80,Journal Of Big Data,0.803
3,Array programming with NumPy,2020,21315,3552.50,Nature,0.800
4,Biomni: A General-Purpose Biomedical AI Agent,2025,65,65.00,bioRxiv (Cold Spring Harbor Laboratory),0.799
5,Array programming with NumPy,2020,18804,3134.00,TUScholarShare (Temple University),0.788
6,A Survey of Convolutional Neural Networks: Ana...,2021,4658,931.60,IEEE Transactions on Neural Networks and Learn...,0.767
7,Qwen3 Technical Report,2025,52,52.00,ArXiv.org,0.766
8,Persona-Driven Benchmarking for Generalizable ...,2022,528,132.00,arXiv (Cornell University),0.759
9,ChatGPT for good? On opportunities and challen...,2023,4545,1515.00,Learning and Individual Differences,0.758


In [4]:
out = DATA_DIR / 'papers_scored.csv'
df_sorted.to_csv(out, index=False)
print(f'Saved → {out.resolve()}')

Saved → /Users/sungjae-cha/Documents/06 아름다운서당/next-seodang/projects/02_research_report_helper/data/papers_scored.csv


## 다음 단계

`03_export_to_claude.ipynb`로 상위 N개를 Claude Desktop에 붙여넣을 markdown 표로 export 하세요.